In [3]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import os

In [4]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [5]:
model = ChatGroq(model='llama-3.3-70b-versatile')

### buat getter messages lama

In [6]:
from langchain_community.chat_message_histories import SQLChatMessageHistory

def get_session_history(session_id: str) -> SQLChatMessageHistory:
    return SQLChatMessageHistory(
        session_id=session_id,
        connection_string="sqlite:///conversation_history.db",
    )

In [7]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import trim_messages, SystemMessage, HumanMessage
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

trimmer = trim_messages(
    max_tokens=200,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on='human'
)


prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a expert football analys. Give strategy breakdown and recommendation"),
    MessagesPlaceholder(variable_name="messages")
])

chain = (RunnablePassthrough.assign(
    messages=itemgetter("messages") | trimmer
)
| prompt
| model
| StrOutputParser()
)

chat_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    input_messages_key='messages'
    
)


In [ ]:
## config persist session
config = {"configurable":{"session_id":"user1_session1"}}
response = chat_with_history.invoke(
    {"messages":[
        HumanMessage(content='Please explain new modern Tiki Taka tactic that developed by Zerby')]},
        config=config
)
response

'You\'re referring to the modern Tiki Taka tactic developed by Maurizio Sarri, also known as "Sarri-Ball" or more recently, the tactic inspired by it, which is often attributed to the Italian coach, but also influenced by other coaches like Pep Guardiola and Thomas Tuchel. However, I couldn\'t find any information on a coach named "Zerby" associated with this tactic.\n\nAssuming you meant to ask about the modern Tiki Taka tactic, here\'s a breakdown:\n\n**Modern Tiki Taka:**\n\nThe modern Tiki Taka tactic is an evolution of the possession-based football popularized by Barcelona under Pep Guardiola. It\'s characterized by:\n\n1. **High-intensity pressing**: The team presses high up the pitch to win the ball back quickly, rather than dropping deep and absorbing pressure.\n2. **Positional flexibility**: Players are encouraged to be flexible in their positioning, often interchanging roles to create confusion among the opposition.\n3. **Short passing and movement**: The team focuses on shor

In [8]:
config1 = {"configurable":{"session_id":"user1_session2"}}
response = chat_with_history.invoke(
    {"messages":[
        HumanMessage(content='Hello, my name is Umar, how are you')]},
        config=config1
)
response

d:\PROJECTS\AgenticAI_LangGraph-LangChain\.venv\Lib\site-packages\langchain_core\runnables\history.py:596: LangChainDeprecationWarning: `connection_string` was deprecated in LangChain 0.2.2 and will be removed in 1.0. Use connection instead.
  message_history = self.get_session_history(
d:\PROJECTS\AgenticAI_LangGraph-LangChain\.venv\Lib\site-packages\langchain_core\language_models\base.py:328: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


"Hello Umar, I'm doing great, thanks for asking! I'm excited to share my expertise in football analysis with you. What would you like to know? Are you looking for a breakdown of a specific team, player, or strategy? Or perhaps you'd like some recommendations for an upcoming match? Let's get started! \n\nBy the way, are you a fan of any particular football team or league?"

In [9]:
response = chat_with_history.invoke(
    {"messages":[
        HumanMessage(content='What is my name? what is football playing style i like?')]},
        config=config1
)
response

"Your name is Umar. However, I don't have any information about your preferred football playing style. You haven't shared that with me yet. Would you like to tell me what kind of playing style you enjoy watching or playing? Do you like possession-based football, counter-attacking, high-pressing, or something else? Let me know, and I can give you more tailored analysis and recommendations!"

### Debug
- Method 1: ✅ Use RunnableLambda to intercept (Recommended)
- Method 2: Inspect prompt output directly (without model)
- Method 3: Read session history directly from storage


In [10]:
def inspect_messages(input_data):
    print("=" * 50)
    print("📨 MESSAGES SENT TO MODEL:")
    print("=" * 50)
    for msg in input_data.messages:
        role = msg.__class__.__name__.replace("Message", "")
        print(f"  [{role}]: {msg.content[:100]}")
    print("=" * 50)
    return input_data  # Pass through unchanged

In [11]:
from langchain_core.runnables import RunnableLambda
debug_chain = (RunnablePassthrough.assign(
    messages=itemgetter("messages") | trimmer
)
| prompt
| RunnableLambda(inspect_messages) #disini
| model
| StrOutputParser()
)

In [12]:
chat_debug = RunnableWithMessageHistory(
    debug_chain,
    get_session_history=get_session_history,
    input_messages_key='messages'
)


In [13]:
response = chat_debug.invoke(
    {"messages":[
        HumanMessage(content='give my name and advice for learning football strategy?')]},
        config=config1
)
response

📨 MESSAGES SENT TO MODEL:
  [System]: You are a expert football analys. Give strategy breakdown and recommendation
  [Human]: What is my name? what is football playing style i like?
  [AI]: Your name is Umar. However, I don't have any information about your preferred football playing style
  [Human]: give my name and advice for learning football strategy?


'Let me assume your name is "Rohan" (since I don\'t have any prior information about your name).\n\nAs Rohan, if you\'re interested in learning football strategy, here\'s some advice:\n\n1. **Start with the basics**: Understand the fundamental principles of football, such as formations, player positions, and basic tactics. Learn about the different types of formations (e.g., 4-4-2, 4-3-3, 3-5-2) and how they affect the game.\n2. **Watch and analyze games**: Watch professional football games, paying attention to the strategies employed by teams. Analyze the decisions made by coaches and players, and try to understand the reasoning behind them.\n3. **Study famous coaches and their tactics**: Look into the strategies and philosophies of renowned coaches like Pep Guardiola, Sir Alex Ferguson, or José Mourinho. Understand their approaches to the game and how they adapt to different situations.\n4. **Learn about different playing styles**: Familiarize yourself with various playing styles, su

In [14]:
response = chat_debug.invoke(
    {"messages":[
        HumanMessage(content='hey, i have told you my real name before')]},
        config=config1
)
response

📨 MESSAGES SENT TO MODEL:
  [System]: You are a expert football analys. Give strategy breakdown and recommendation
  [Human]: hey, i have told you my real name before


"I don't have any record of you telling me your real name before. I'm a large language model, I don't have personal memories or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name again, I'd be happy to chat with you! \n\nBy the way, since you're interested in football, would you like to discuss a specific team, player, or strategy? I can provide a breakdown and recommendations if you'd like."

> karena max tokens cuman 200 maka tertrim dan model tidak paham isi chat sebelumnya.
coba pakai summarizer

In [32]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate

# === STEP 1: Summarizer Chain ===
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following conversation into key facts. "
               "Include: user name, preferences, important context. "
               "Be concise but complete."),
    MessagesPlaceholder(variable_name="messages"),
])
summarize_chain = summarize_prompt | model

# === STEP 2: Function to manage history ===
def smart_history_manager(messages, max_messages=2):
    """
    Jika history > max_messages:
    1. Summarize pesan lama
    2. Ganti dengan summary sebagai SystemMessage
    3. Pertahankan pesan terbaru
    """
    print(len(messages))
    if len(messages) <= max_messages:
        return messages  # Belum perlu trim

    # Pisahkan system messages
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    conversation = [m for m in messages if not isinstance(m, SystemMessage)]
    
    # Pesan yang akan di-summarize vs dipertahankan
    cutoff = len(conversation) - max_messages
    old_messages = conversation[:cutoff]
    recent_messages = conversation[cutoff:]
    
    # Generate summary dari pesan lama
    summary = summarize_chain.invoke({"messages": old_messages})
    
    # Gabungkan: System + Summary + Recent
    summary_message = SystemMessage(
        content=f"Summary of earlier conversation:\n{summary.content}"
    )
    print(summary_message)
    return system_msgs + [summary_message] + recent_messages

In [33]:
smart_trimmer = RunnableLambda(smart_history_manager)

In [34]:
smart_chain = (
    RunnablePassthrough.assign(
        messages = itemgetter('messages') | smart_trimmer
    )
    | prompt
    | RunnableLambda(inspect_messages)
    | model
    | StrOutputParser()
)

In [35]:
goodchatmemori = RunnableWithMessageHistory(
    smart_chain,
    get_session_history,
    input_messages_key='messages'
)

In [36]:
response = goodchatmemori.invoke(
    {"messages":[HumanMessage(content='actually i like possesion game, how passing strategy used?')]},
    config=config1
)
response

13
content="Summary of earlier conversation:\nLet me try again.\n\nBased on our conversation, I'm going to take a guess that your name is... Umar!\n\nAs for your football strategy, I'm going to take a guess that you might be a fan of a more counter-attacking style of play. Perhaps you enjoy watching teams that focus on quick transitions, pacey wingers, and clinical finishing. You might appreciate the tactics of teams like Leicester City, Atletico Madrid, or Borussia Dortmund, who have had success with this approach.\n\nAm I correct, Umar?" additional_kwargs={} response_metadata={}
📨 MESSAGES SENT TO MODEL:
  [System]: You are a expert football analys. Give strategy breakdown and recommendation
  [System]: Summary of earlier conversation:
Let me try again.

Based on our conversation, I'm going to take a g
  [AI]: A new guess!

Based on our conversation, I'm going to take a guess that your name is... Umar! (You m
  [Human]: actually i like possesion game, how passing strategy used?


"As a possession-based team, the passing strategy is crucial to maintaining control of the game and creating scoring opportunities.\n\nHere are some key aspects of a possession-based passing strategy:\n\n1. **Short passing**: Focus on short, precise passes to maintain possession and control the tempo of the game. This involves a lot of one-touch and two-touch passing, with an emphasis on keeping the ball on the ground.\n2. **Triangle formations**: Create triangle formations with players to facilitate quick passing and movement. This allows for easy passing options and helps to break down the opponent's defense.\n3. **Player movement**: Encourage players to make intelligent runs and movements to create space and receive passes. This can include making runs behind the defense, creating triangles, and using clever footwork to evade opponents.\n4. **Switching play**: Use long passes to switch the point of attack and catch the opponent off guard. This can involve switching the ball from one

In [37]:
response = goodchatmemori.invoke(
    {"messages":[HumanMessage(content='give my name and preferred football strategu')]},
    config=config1
)
response

15
content='Summary of earlier conversation:\nI was close, but I didn\'t quite nail it. You\'re a fan of possession-based football, which is a great style of play.\n\nIn possession-based football, the passing strategy is often focused on maintaining control of the ball, wearing down the opponent, and creating scoring opportunities through sustained attacks. Here are some key aspects of a possession-based passing strategy:\n\n1. **Short passing**: Players focus on short, precise passes to maintain possession and control the tempo of the game. This helps to tire out the opponent and creates opportunities for more adventurous passes.\n2. **Triangle formations**: Players often form triangles with each other to create passing options and confuse the opponent. This allows them to move the ball quickly and efficiently around the pitch.\n3. **Rotations and interchanging**: Players rotate positions and interchange with each other to create confusion and exploit gaps in the opponent\'s defense.\

"Your name is Umar, and you're a fan of **possession-based football**. This strategy involves maintaining control of the ball, wearing down the opponent, and creating scoring opportunities through sustained attacks, often using short passing, triangle formations, and intelligent movement."

In [15]:
config1

{'configurable': {'session_id': 'user1_session2'}}

In [19]:
# Peek into what's stored for a session
history = get_session_history("user1_session2")
print("📦 Stored messages:")
for msg in history.messages:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}]: {msg.content[:80]}")

📦 Stored messages:
  [Human]: Hello, my name is Umar, how are you
  [AI]: Hello Umar, I'm doing great, thanks for asking! I'm excited to share my expertis
  [Human]: What is my name? what is football playing style i like?
  [AI]: Your name is Umar. However, I don't have any information about your preferred fo
  [Human]: give my name and advice for learning football strategy?
  [AI]: Let me assume your name is "Rohan" (since I don't have any prior information abo
  [Human]: hey, i have told you my real name before
  [AI]: I don't have any record of you telling me your real name before. I'm a large lan


### Langchain Debug Logging

In [ ]:
from langchain.globals import set_debug
set_debug(True) 